# 🤖 Phase 4: Model Training, Class Imbalance Handling & Threshold Tuning

**Penelitian**: *Predicting Flaky Test in Cross-Project Scenario*

Notebook ini mengeksekusi **Phase 4** dari roadmap penelitian flaky test:
1. **Pemilihan Algoritma Baseline & Advanced**:
   - **Baseline**: *Logistic Regression*, *Naive Bayes (GaussianNB)*, dan *Dummy Classifier (Stratified & Most Frequent)*.
   - **Tree-based Ensembles**: *Random Forest*, *XGBoost*, *LightGBM*, dan *CatBoost*.
2. **Penanganan Imbalance (Extreme Class Imbalance ~3.65% Flaky)**:
   - **Cost-Sensitive Learning**: Menggunakan parameter `scale_pos_weight` pada XGBoost/LightGBM/CatBoost dan `class_weight='balanced'` pada Random Forest & Logistic Regression.
   - **Resampling Strategy**: *SMOTE (Synthetic Minority Over-sampling Technique)* diterapkan **HANYA pada data Training** (tidak pernah menyentuh data Test untuk mencegah leakage).
3. **Threshold Tuning (Optimal Decision Boundary)**:
   - Mengoptimalkan threshold probabilitas (menggantikan default `0.5`) yang memaksimalkan **F1-Score** pada data training/internal validation.
4. **Cross-Project LOOCV & Multi-Project Evaluation**:
   - Melatih dan menguji model pada skenario **Leave-One-Project-Out (LOOCV / LOPO-CV)** 24 fold serta skenario Multi-Project Split (70/30).
5. **Ekspor Hasil & Ringkasan Metrik**: Menyimpan hasil prediksi dan komparasi performa model ke `data/model_comparison_results.csv`.

## 1. Import Library & Load Engineered Dataset

In [1]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Algoritma ML
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Resampling & Metrics
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    average_precision_score, precision_recall_curve, confusion_matrix
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)

# Load dataset & metadata split Phase 3
df = pd.read_csv("../data/engineered_test_features.csv")
with open("../data/validation_splits_metadata.json") as f:
    splits_meta = json.load(f)

print(f"Dataset Shape: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"Total Proyek: {df['project'].nunique()}")
print(f"Total Flaky: {df['flaky'].sum()} ({df['flaky'].mean()*100:.2f}%)")

Dataset Shape: 22236 baris, 58 kolom
Total Proyek: 24
Total Flaky: 811 (3.65%)


## 2. Seleksi Fitur Project-Agnostic untuk Model Training

Fitur input $X$ dirancang sepenuhnya **Project-Agnostic** (tanpa identifier proyek, nama test, atau nama kelas):
- **Test Smells** (8 fitur binary)
- **Test Metrics & Coverage Ter-Normalisasi** (9 fitur ber-skala `RobustScaler` & rasio)
- **Code Churn PCA** (3 komponen utama PCA dari 8 window `hIndex`)

Total 20 fitur prediktif terpilih.

In [2]:
smell_features = [
    'assertion-roulette', 'conditional-test-logic', 'eager-test', 'fire-and-forget',
    'indirect-testing', 'mystery-guest', 'resource-optimism', 'test-run-war'
]

scaled_features = [
    'rob_log_ExecutionTime', 'rob_log_testLength', 'rob_log_numCoveredLines',
    'rob_log_projectSourceLinesCovered', 'rob_log_projectSourceClassesCovered',
    'rob_coverage_ratio', 'rob_assert_density', 'rob_class_coverage_ratio',
    'rob_num_third_party_libs'
]

pca_churn_features = ['churn_pca_1', 'churn_pca_2', 'churn_pca_3']

selected_features = smell_features + scaled_features + pca_churn_features
print(f"Total Selected Project-Agnostic Features: {len(selected_features)}")
print(selected_features)

Total Selected Project-Agnostic Features: 20
['assertion-roulette', 'conditional-test-logic', 'eager-test', 'fire-and-forget', 'indirect-testing', 'mystery-guest', 'resource-optimism', 'test-run-war', 'rob_log_ExecutionTime', 'rob_log_testLength', 'rob_log_numCoveredLines', 'rob_log_projectSourceLinesCovered', 'rob_log_projectSourceClassesCovered', 'rob_coverage_ratio', 'rob_assert_density', 'rob_class_coverage_ratio', 'rob_num_third_party_libs', 'churn_pca_1', 'churn_pca_2', 'churn_pca_3']


## 3. Modul Evaluasi & Threshold Tuning

Pada data dengan ketimpangan ekstrem, threshold probabilitas default (`0.5`) sering menghasilkan recall yang sangat rendah atau nol.
Fungsi `find_optimal_threshold` mengidentifikasi threshold probabilitas yang memaksimalkan F1-Score pada data train.

In [3]:
def find_optimal_threshold(y_true, y_probs):
    """Mencari threshold probabilitas yang memaksimalkan F1-Score."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    # Hindari pembagian dengan nol
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    optimal_idx = np.argmax(f1_scores[:-1]) if len(thresholds) > 0 else 0
    optimal_threshold = thresholds[optimal_idx] if len(thresholds) > 0 else 0.5
    return optimal_threshold, f1_scores[optimal_idx]

def evaluate_predictions(y_true, y_probs, threshold=0.5):
    """Menghitung metrik performa klasifikasi."""
    y_pred = (y_probs >= threshold).astype(int)
    
    # Hitung ROC-AUC dan PR-AUC (jika ada lebih dari 1 kelas pada y_true)
    if len(np.unique(y_true)) > 1:
        roc_auc = roc_auc_score(y_true, y_probs)
        pr_auc = average_precision_score(y_true, y_probs)
    else:
        roc_auc = 0.5
        pr_auc = 0.0
        
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    return {
        'Precision': round(p, 4),
        'Recall': round(r, 4),
        'F1-Score': round(f1, 4),
        'ROC-AUC': round(roc_auc, 4),
        'PR-AUC': round(pr_auc, 4)
    }

## 4. Eksperimen Multi-Project Split (70% Train Projects, 30% Test Projects)

Membandingkan model **Baseline** vs **Tree-based Ensembles** dengan dua pendekatan penanganan imbalance:
1. **Cost-Sensitive Learning** (`class_weight='balanced'` / `scale_pos_weight`).
2. **Resampling** (*SMOTE* pada data training).
3. **Threshold Tuning** (Threshold optimal vs default 0.5).

In [4]:
# Pisahkan Train & Test berdasarkan konfigurasi Phase 3
multi_split = splits_meta['multi_project_split_70_30']
train_projects = multi_split['train_projects']
test_projects = multi_split['test_projects']

train_mask = df['project'].isin(train_projects)
test_mask = df['project'].isin(test_projects)

X_train = df.loc[train_mask, selected_features].values
y_train = df.loc[train_mask, 'flaky'].values
X_test = df.loc[test_mask, selected_features].values
y_test = df.loc[test_mask, 'flaky'].values

scale_pos = (len(y_train) - y_train.sum()) / (y_train.sum() + 1e-5)
print(f"Train Data: {X_train.shape[0]} samples, Flaky: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Test Data : {X_test.shape[0]} samples, Flaky: {y_test.sum()} ({y_test.mean()*100:.2f}%)")
print(f"scale_pos_weight: {scale_pos:.2f}")

Train Data: 18221 samples, Flaky: 617 (3.39%)
Test Data : 4015 samples, Flaky: 194 (4.83%)
scale_pos_weight: 28.53


In [5]:
# Definisi Algoritma yang diuji
models = {
    'Dummy (Most Frequent)': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression (Balanced)': LogisticRegression(class_weight='balanced', max_iter=500, random_state=42),
    'Gaussian Naive Bayes': GaussianNB(),
    'Random Forest (Balanced)': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost (Cost-Sensitive)': XGBClassifier(scale_pos_weight=scale_pos, eval_metric='logloss', random_state=42, n_jobs=-1),
    'LightGBM (Cost-Sensitive)': LGBMClassifier(scale_pos_weight=scale_pos, random_state=42, n_jobs=-1, verbose=-1),
    'CatBoost (Cost-Sensitive)': CatBoostClassifier(scale_pos_weight=scale_pos, iterations=200, verbose=0, random_state=42)
}

benchmark_results = []

for model_name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - t0
    
    # Probabilitas train untuk threshold tuning
    train_probs = model.predict_proba(X_train)[:, 1] if hasattr(model, 'predict_proba') else model.predict(X_train)
    test_probs = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else model.predict(X_test)
    
    opt_thresh, opt_train_f1 = find_optimal_threshold(y_train, train_probs)
    
    # Evaluasi dengan threshold 0.5
    eval_default = evaluate_predictions(y_test, test_probs, threshold=0.5)
    # Evaluasi dengan optimal threshold
    eval_opt = evaluate_predictions(y_test, test_probs, threshold=opt_thresh)
    
    benchmark_results.append({
        'Model': model_name,
        'Train Time (s)': round(fit_time, 2),
        'ROC-AUC': eval_default['ROC-AUC'],
        'PR-AUC': eval_default['PR-AUC'],
        'F1 (Thresh=0.5)': eval_default['F1-Score'],
        'Recall (Thresh=0.5)': eval_default['Recall'],
        'Opt Thresh': round(opt_thresh, 4),
        'F1 (Tuned)': eval_opt['F1-Score'],
        'Recall (Tuned)': eval_opt['Recall']
    })

df_benchmark = pd.DataFrame(benchmark_results).sort_values(by='PR-AUC', ascending=False)
print("=== Perbandingan Model pada Skenario Multi-Project (70/30) ===")
df_benchmark

=== Perbandingan Model pada Skenario Multi-Project (70/30) ===


,Model,Train Time (s),ROC-AUC,PR-AUC,F1 (Thresh=0.5),Recall (Thresh=0.5),Opt Thresh,F1 (Tuned),Recall (Tuned)
5,LightGBM (Cost-Sensitive),0.44,0.7326,0.0908,0.0621,0.0722,0.8164,0.0078,0.0052
2,Gaussian Naive Bayes,0.01,0.5623,0.0787,0.1456,0.4845,0.9991,0.1818,0.3144
6,CatBoost (Cost-Sensitive),1.26,0.6717,0.0737,0.0436,0.0412,0.8539,0.0090,0.0052
3,Random Forest (Balanced),0.50,0.6320,0.0719,0.0102,0.0052,0.4900,0.0102,0.0052
4,XGBoost (Cost-Sensitive),0.81,0.6565,0.0677,0.0311,0.0258,0.9122,0.0000,0.0000
1,Logistic Regression (Balanced),0.24,0.5896,0.0615,0.0971,0.3763,0.8571,0.1013,0.1856
0,Dummy (Most Frequent),0.00,0.5000,0.0483,0.0000,0.0000,0.0000,0.0922,1.0000


### 4.1 Dampak Resampling SMOTE pada Data Training

Menguji dampak penambahan sampel sintetis (*SMOTE*) pada subset data train (tanpa mengubah data test).

In [6]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print(f"Distribusi Train setelah SMOTE: Kelas 0 = {(y_train_smote == 0).sum()}, Kelas 1 = {(y_train_smote == 1).sum()}")

# Uji Random Forest & LightGBM dengan SMOTE
smote_models = {
    'Random Forest + SMOTE': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'LightGBM + SMOTE': LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)
}

smote_results = []
for name, model in smote_models.items():
    model.fit(X_train_smote, y_train_smote)
    test_probs = model.predict_proba(X_test)[:, 1]
    eval_res = evaluate_predictions(y_test, test_probs, threshold=0.5)
    eval_res['Model'] = name
    smote_results.append(eval_res)

df_smote = pd.DataFrame(smote_results)[['Model', 'ROC-AUC', 'PR-AUC', 'Precision', 'Recall', 'F1-Score']]
df_smote

Distribusi Train setelah SMOTE: Kelas 0 = 17604, Kelas 1 = 17604


,Model,ROC-AUC,PR-AUC,Precision,Recall,F1-Score
0,Random Forest + SMOTE,0.5929,0.0612,0.0755,0.0206,0.0324
1,LightGBM + SMOTE,0.7322,0.1149,0.0152,0.0155,0.0153


## 5. Pelatihan & Evaluasi Cross-Project LOOCV (24-Fold)

Mengevaluasi algoritma ensemble terbaik (**LightGBM**, **Random Forest**, dan **XGBoost**) pada seluruh 24 fold *Leave-One-Project-Out* untuk mengukur generalisasi nyata ke proyek unseen.

In [7]:
loocv_folds = splits_meta['loocv_folds']
loocv_results = []

print(f"Memulai evaluasi LOOCV untuk {len(loocv_folds)} fold...")

for proj_name, fold_info in loocv_folds.items():
    fold_idx = fold_info['fold']
    
    train_mask = (df['project'] != proj_name)
    test_mask = (df['project'] == proj_name)
    
    X_tr = df.loc[train_mask, selected_features].values
    y_tr = df.loc[train_mask, 'flaky'].values
    X_te = df.loc[test_mask, selected_features].values
    y_te = df.loc[test_mask, 'flaky'].values
    
    scale_pos = (len(y_tr) - y_tr.sum()) / (y_tr.sum() + 1e-5)
    
    # Model LightGBM Cost-Sensitive
    clf = LGBMClassifier(scale_pos_weight=scale_pos, random_state=42, n_jobs=-1, verbose=-1)
    clf.fit(X_tr, y_tr)
    
    probs = clf.predict_proba(X_te)[:, 1]
    
    # Evaluasi
    n_flaky = int(y_te.sum())
    roc = roc_auc_score(y_te, probs) if n_flaky > 0 and n_flaky < len(y_te) else np.nan
    pr = average_precision_score(y_te, probs) if n_flaky > 0 else np.nan
    
    # Optimal threshold dari train set
    tr_probs = clf.predict_proba(X_tr)[:, 1]
    opt_th, _ = find_optimal_threshold(y_tr, tr_probs)
    preds = (probs >= opt_th).astype(int)
    
    loocv_results.append({
        'Fold': fold_idx,
        'Test Project': proj_name,
        'Test Samples': len(y_te),
        'Flaky Samples': n_flaky,
        'ROC-AUC': round(roc, 4) if not np.isnan(roc) else 'N/A',
        'PR-AUC': round(pr, 4) if not np.isnan(pr) else 'N/A',
        'F1-Tuned': round(f1_score(y_te, preds, zero_division=0), 4),
        'Recall': round(recall_score(y_te, preds, zero_division=0), 4),
        'Precision': round(precision_score(y_te, preds, zero_division=0), 4)
    })

df_loocv_results = pd.DataFrame(loocv_results)
print("=== Hasil Evaluasi 24 Fold LOOCV (LightGBM Cost-Sensitive) ===")
df_loocv_results

Memulai evaluasi LOOCV untuk 24 fold...
=== Hasil Evaluasi 24 Fold LOOCV (LightGBM Cost-Sensitive) ===


,Fold,Test Project,Test Samples,Flaky Samples,ROC-AUC,PR-AUC,F1-Tuned,Recall,Precision
0,1,achilles,1317,4,0.7258,0.0068,0.0000,0.0000,0.0000
1,2,activiti,2044,32,0.6287,0.0273,0.0267,0.0312,0.0233
2,3,alluxio,187,116,0.8255,0.8754,0.0000,0.0000,0.0000
3,4,ambari,324,52,0.7792,0.3201,0.2963,0.3077,0.2857
4,5,assertj-core,6261,1,0.3676,0.0003,0.0000,0.0000,0.0000
5,6,commons-exec,55,1,1.0,1.0,0.6667,1.0000,0.5000
6,7,elastic-job-lite,558,3,0.842,0.0216,0.0000,0.0000,0.0000
7,8,handlebars.java,427,1,0.8239,0.0132,0.0000,0.0000,0.0000
8,9,hbase,431,145,0.7454,0.6258,0.0272,0.0138,1.0000
9,10,hector,142,33,0.6494,0.3049,0.0000,0.0000,0.0000


## 6. Simpan Hasil Perbandingan Model (Phase 4 Output)

In [8]:
benchmark_out_path = "../data/model_comparison_results.csv"
loocv_out_path = "../data/loocv_lgbm_results.csv"

df_benchmark.to_csv(benchmark_out_path, index=False)
df_loocv_results.to_csv(loocv_out_path, index=False)

print(f"✅ Hasil komparasi model disimpan ke: {benchmark_out_path}")
print(f"✅ Hasil per-fold LOOCV disimpan ke: {loocv_out_path}")

✅ Hasil komparasi model disimpan ke: ../data/model_comparison_results.csv
✅ Hasil per-fold LOOCV disimpan ke: ../data/loocv_lgbm_results.csv
